# BigAlpha Step07：Step02 完整公榜训练期（2019–2024）

## Goal

本轮是严格的单变量实验：保持 Step02 的网络结构、输入字段、64根1分钟序列、600只固定抽样股票、损失权重、训练轮数与随机种子不变，只把训练结束日期从 `2023-12-31` 延长到 `2024-12-31`。

首次点击“全部运行”会从零训练并生成独立权重 `step_next_07_step02_train2019_2024_model.json`。再次运行或提交时，如果同目录已经存在该JSON，则只回读检查，不重复训练。

## Setup

### 1. 定义比赛推理入口

后台调用 `main(datasources, start_date, end_date)` 时，只加载同目录JSON并对隐藏区间推理。

In [1]:
def main(datasources, start_date, end_date):
    import numpy as np
    from step_next_07_step02_train2019_2024 import main as model_main

    score_data = model_main(datasources, start_date, end_date)
    required_columns = ['date', 'instrument', 'score']
    if list(score_data.columns) != required_columns:
        raise ValueError(
            '模型输出列必须严格为 {}，实际为 {}'.format(
                required_columns, list(score_data.columns)
            )
        )
    if score_data.empty:
        raise ValueError('模型输出为空')
    if score_data.duplicated(['date', 'instrument']).any():
        raise ValueError('模型输出存在重复 date/instrument')
    if not np.isfinite(score_data['score'].to_numpy(dtype=np.float64)).all():
        raise ValueError('模型输出存在 NaN/inf')
    return score_data


## Steps

### 2. 检查单变量边界

In [2]:
import importlib
import os
import sys
import time

import pandas as pd
import torch
from IPython.display import display

import step_next_07_step02_train2019_2024 as experiment
importlib.reload(experiment)

assert experiment.TRAIN_START == '2019-01-01 00:00:00'
assert experiment.TRAIN_END == '2024-12-31 23:59:59'
assert experiment.INSTRUMENT_SELECTION_END == '2023-12-31 23:59:59'
assert experiment.MAX_TRAIN_INSTRUMENTS == 600
assert experiment.SEQ_LEN == 64
assert experiment.EPOCHS == 5
assert experiment.SEED == 20260719
assert experiment.MSE_WEIGHT == 0.25
assert experiment.DAILY_IC_WEIGHT == 0.75
assert experiment.PAIRWISE_WEIGHT == 0.05
assert experiment.MODEL_CFG['relation_topk'] == 32

model = experiment.DailyCrossSectionTransformer(**experiment.MODEL_CFG)
parameter_count = experiment.validate_parameter_count(model)
assert parameter_count == 151619

print({
    'python': sys.version.replace('\n', ' '),
    'torch': torch.__version__,
    'gpu_available': torch.cuda.is_available(),
    'experiment_id': experiment.EXPERIMENT_ID,
    'parameters': parameter_count,
    'train_range': (experiment.TRAIN_START, experiment.TRAIN_END),
    'instrument_selection_range': (
        experiment.TRAIN_START,
        experiment.INSTRUMENT_SELECTION_END,
    ),
    'max_train_instruments': experiment.MAX_TRAIN_INSTRUMENTS,
    'model_path': experiment.MODEL_PATH,
})
print('PASS: 仅扩展训练结束日期，Step02其余核心配置保持不变')

{'python': '3.11.8 (main, Apr 23 2026, 12:30:59) [GCC 11.5.0 20240719 (Red Hat 11.5.0-11)]', 'torch': '2.3.0+cu121', 'gpu_available': True, 'experiment_id': 'step_next_07_step02_train2019_2024', 'parameters': 151619, 'train_range': ('2019-01-01 00:00:00', '2024-12-31 23:59:59'), 'instrument_selection_range': ('2019-01-01 00:00:00', '2023-12-31 23:59:59'), 'max_train_instruments': 600, 'model_path': '/home/aiuser/work/提交/step_next_07_step02_train2019_2024_model.json'}
PASS: 仅扩展训练结束日期，Step02其余核心配置保持不变


### 3. 首次运行时从零训练，已有JSON时只做回读检查

请使用GPU Notebook。公榜阶段提交三个文件时JSON已经存在，因此不会在推理任务中重复训练。

In [3]:
datasources = {'bar1m': 'bigalpha_2026_stock_bar1m'}
TRAIN_IF_MODEL_MISSING = True
trained_now = False

if not os.path.isfile(experiment.MODEL_PATH):
    if not TRAIN_IF_MODEL_MISSING:
        raise FileNotFoundError(
            '缺少权重且已禁止训练：{}'.format(experiment.MODEL_PATH)
        )
    assert torch.cuda.is_available(), '首次训练请切换到GPU Notebook'
    torch.cuda.empty_cache()
    training_started = time.time()
    print('开始从零训练 Step07：Step02 2019–2024完整公榜训练期……')
    model_path = experiment.train_and_save(datasources)
    trained_now = True
    training_minutes = round((time.time() - training_started) / 60.0, 2)
else:
    model_path = experiment.MODEL_PATH
    training_minutes = 0.0
    print('检测到已有Step07 JSON，跳过训练：', model_path)

checkpoint = experiment.load_model(model_path, map_location='cpu')
experiment.validate_checkpoint(checkpoint)
assert checkpoint['train_start'] == experiment.TRAIN_START
assert checkpoint['train_end'] == experiment.TRAIN_END
assert checkpoint['instrument_selection_end'] == experiment.INSTRUMENT_SELECTION_END
assert int(checkpoint['parameter_count']) == parameter_count
assert len(checkpoint['train_instruments']) == 600

print({
    'status': 'STEP07_FULL_PUBLIC_TRAIN_READY',
    'trained_now': trained_now,
    'model_path': os.path.abspath(model_path),
    'model_size_mb': round(os.path.getsize(model_path) / 1024 ** 2, 2),
    'training_minutes': training_minutes,
    'train_start': checkpoint['train_start'],
    'train_end': checkpoint['train_end'],
    'train_instruments': len(checkpoint['train_instruments']),
    'data_info': checkpoint.get('data_info'),
})
display(pd.DataFrame(checkpoint['epoch_history']))

开始从零训练 Step07：Step02 2019–2024完整公榜训练期……
[2026-07-24 09:57:50] [info     ] 开始训练相似股票图注意力模型                 device=cuda instruments=600 table=bigalpha_2026_stock_bar1m train_end='2024-12-31 23:59:59' train_start='2019-01-01 00:00:00'
[2026-07-24 09:59:12] [info     ] 相似股票图训练数据分块完成                  chunk=1/4 samples=119455
[2026-07-24 10:00:16] [info     ] 相似股票图训练数据分块完成                  chunk=2/4 samples=128504
[2026-07-24 10:01:01] [info     ] 相似股票图训练数据分块完成                  chunk=3/4 samples=117536
[2026-07-24 10:01:38] [info     ] 相似股票图训练数据分块完成                  chunk=4/4 samples=103966
[2026-07-24 10:01:47] [info     ] 相似股票图训练集构建完成                   days=1455 elapsed=236.5 instruments=599 max_daily_samples=341 min_daily_samples=272 samples=469461 x_mb=1146.15
[2026-07-24 10:02:15] [info     ] 相似股票图注意力 epoch 完成              cross_gate=0.5004530549049377 daily_ic=0.07180035713755228 elapsed_seconds=27.35 epoch=1 ic_days=1455 mse=0.34041140762626804 negative_ic_days=444 pairs=186066 pairwis

,epoch,total_loss,mse,daily_ic,negative_ic_days,ic_days,pairwise,pairs,steps,cross_gate,relation_gate,elapsed_seconds
0,1,0.815843,0.340411,0.071800,444,1455,0.691810,186066,364,0.500453,0.117706,27.35
1,2,0.800613,0.336119,0.090575,370,1455,0.690296,186042,364,0.496242,0.116677,28.63
2,3,0.797253,0.334761,0.094594,341,1455,0.690166,186049,364,0.492220,0.115728,29.41
3,4,0.794768,0.334038,0.097642,349,1455,0.689797,186065,364,0.489228,0.115688,28.52
4,5,0.792829,0.333214,0.099940,322,1455,0.689612,186057,364,0.485018,0.114909,29.03


## Checks

### 4. 提交文件检查

2024已经进入训练集，因此本Notebook不会调用2024官方评估并把它误称为样本外结果。模型效果应通过新的公榜隐藏区间提交记录，与原0.73276基准比较。

In [4]:
module_dir = os.path.dirname(os.path.abspath(experiment.__file__))
required_runtime_files = [
    'step_next_07_step02_train2019_2024.py',
    'step_next_07_step02_train2019_2024_model.json',
]
missing_runtime_files = [
    name for name in required_runtime_files
    if not os.path.isfile(os.path.join(module_dir, name))
]
assert not missing_runtime_files, (
    '运行目录缺少文件：{}'.format(missing_runtime_files)
)
print('PASS: Python模块与JSON运行文件齐全')
print('提交时还需同时选择本Notebook文件')
print('STEP_NEXT_07_TRAIN2019_2024_DONE')
display(pd.Series({
    'experiment': experiment.EXPERIMENT_ID,
    'train_start': experiment.TRAIN_START,
    'train_end': experiment.TRAIN_END,
    'model_json': os.path.basename(experiment.MODEL_PATH),
    'next_validation': '提交公榜隐藏区间，与0.73276基准比较',
}, name='Step07_Submission_Ready'))

PASS: Python模块与JSON运行文件齐全
提交时还需同时选择本Notebook文件
STEP_NEXT_07_TRAIN2019_2024_DONE


experiment                    step_next_07_step02_train2019_2024
train_start                                  2019-01-01 00:00:00
train_end                                    2024-12-31 23:59:59
model_json         step_next_07_step02_train2019_2024_model.json
next_validation                            提交公榜隐藏区间，与0.73276基准比较
Name: Step07_Submission_Ready, dtype: object

## Next Steps

将本Notebook、同名Python模块和刚生成的JSON原封不动放在同一目录提交。不要覆盖或删除原Step02提交；两次提交会作为独立记录保留。返回新的公榜总分以及IC Mean、IC IR、Sharpe、Stress四项全场分位后，再决定是否进入完整股票池实验。